# Part 6: Training with Google Cloud Platform (GCP)

This notebook provides an overview of how to train your TensorFlow models in the cloud using Google Cloud Platform (specifically Vertex AI, the modern successor to AI Platform).

### Why train on GCP?
- **Scalability**: Access powerful GPUs (NVIDIA T4, V100, A100) and TPUs that might not be available locally.
- **Managed Infrastructure**: Google handles the provisioning and scaling of resources.
- **Experiment Tracking**: Easily track different training runs and hyperparameter tuning jobs.

## 1. Prerequisites

Before you begin, ensure you have:
1. A GCP Project.
2. Billing enabled.
3. Vertex AI API enabled.
4. Google Cloud SDK installed locally (optional but recommended).
5. A Cloud Storage (GCS) bucket to store your data and model artifacts.

In [ ]:
# Import necessary libraries
import os
import tensorflow as tf
from google.cloud import aiplatform

# Set your project ID and bucket name
PROJECT_ID = "your-project-id"
BUCKET_NAME = "your-bucket-name"
REGION = "us-central1"

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=f"gs://{BUCKET_NAME}")

## 2. Preparing the Training Script

To train on GCP, you usually package your code into a Python script. This script will be executed on a managed VM.

In [ ]:
%%writefile task.py
import tensorflow as tf
import argparse
import os

def train_model(args):
    # 1. Create a simple model (same as Part 4)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(1,))
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')

    # 2. Dummy data for demonstration
    import numpy as np
    X = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=float)
    Y = np.array([10.0, 20.0, 30.0, 40.0, 50.0], dtype=float)

    # 3. Train
    model.fit(X, Y, epochs=args.epochs)

    # 4. Save the model to GCS
    model.save(args.model_dir)
    print(f"Model saved to {args.model_dir}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--model-dir', type=str, default=os.getenv('AIP_MODEL_DIR'))
    args = parser.parse_args()
    train_model(args)

## 3. Submitting a Custom Training Job

We use the `aiplatform.CustomTrainingJob` to submit our `task.py` to Vertex AI.

In [ ]:
# Define the job
job = aiplatform.CustomTrainingJob(
    display_name="my_tf_training_job",
    script_path="task.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-12.py310:latest",
    requirements=["tensorflow==2.12.0"]
)

# Run the job
# model = job.run(
#     args=["--epochs=50"],
#     replica_count=1,
#     machine_type="n1-standard-4",
#     base_output_dir=f"gs://{BUCKET_NAME}/output"
# )

## 4. Monitoring the Job

Once submitted, you can monitor the job in the [GCP Console](https://console.cloud.google.com/vertex-ai/training/training-pipelines). You will see logs, resource utilization, and the status of your training.

### Key concepts to explore next:
- **Hyperparameter Tuning**: Using Vertex AI Vizier to find the best parameters.
- **Distributed Training**: Using `tf.distribute.Strategy` to train across multiple GPUs/Nodes.
- **Vertex AI Pipelines**: Orchestrating complex ML workflows.